In [31]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from tqdm.autonotebook import tqdm

In [32]:
def initial(N):
    # Определяем диапазон x
    x = np.linspace(0, 2, N)

    # Параметры для гауссовских распределений
    # Кривая 1: широкий и низкий пик
    mu1 = 1.0
    sigma1 = 0.12
    A1 = 0.38
    u0 = A1 * np.exp(-((x - mu1) ** 2) / (2 * sigma1 ** 2))

    # Кривая 2: узкий и высокий пик
    mu2 = 1.75
    sigma2 = 0.12
    A2 = 0.5
    v0 = A2 * np.exp(-((x - mu2) ** 2) / (2 * sigma2 ** 2))
    return u0, v0
    
def func_p(N):
    x = np.array([0.0, 0.1, 0.5, 0.8, 0.91, 1.0, 1.1, 1.5, 2.0])
    y = np.array([0.1, 0.5, 1.0, 0.6, 0.3, 0.3, 0.3, 0.6, 0.1])

    # Аппроксимация с помощью кубического сплайна
    x_new = np.linspace(0, 2, N)
    spline = interp1d(x, y, kind='cubic')
    y_new = spline(x_new)
    return y_new

In [33]:
k1 = 0.03
k2 = 0.04
gamma1 = 0.01
gamma2 = 4/3 * gamma1
def E1(u,v):
    return k1 + gamma1 * u * v
def E2(u,v):
    return k2 + gamma2 * u * v
alpha1 = 0
alpha2 = 0
beta1 = 0
beta2 = 0
eta1 = 3
eta2 = 4



T = 20
L = 2
tau = 0.01
h = 0.01
M = int(T / tau)
N = int(L / h)

u = np.zeros((M+1, N+1))
v = np.zeros((M+1, N+1))
u[0][:], v[0][:] = initial(N+1)
p = func_p(N+1)

pbar = tqdm(range(M), desc="Calculating")
for j in range(M):
    alpha = np.zeros((N+1))
    beta = np.zeros((N+1))
    alpha[1] = 0
    beta[1] = 0
    for i in range(1,N):
        e1 = -E1((u[j,i] + u[j,i+1]) / 2, (v[j,i] + v[j,i+1]) / 2)
        e1_ = -E1((u[j,i] + u[j,i-1]) / 2, (v[j,i] + v[j,i-1]) / 2)

        A = -1/h * (e1_ / h - alpha1/2 * (p[i]-p[i-1])/h - beta1/2 * (v[j][i]-v[j][i-1])/h )
        B = -1/h * (-e1 / h + alpha1/2 * (p[i+1]-p[i])/h + beta1/2 * (v[j][i+1]-v[j][i])/h - e1_ / h - alpha1/2 * (p[i]-p[i-1])/h + beta1/2 * (v[j][i]-v[j][i-1])/h) - 1/tau
        C = -1/h * (e1 / h + alpha1 /2 * (p[i+1]-p[i])/h + beta1/2 * (v[j][i+1]-v[j][i]/h))
        alpha[i+1] = -C / (A * alpha[i] + B)
        beta[i+1] = (-u[j][i]/tau - eta1*u[j][i]*(1 - (u[j][i] + v[j][i])/p[i]) - A * beta[i]) / (A * alpha[i] + B)

    u[j+1,N] = 0
    for i in range(N-1,0,-1):
        u[j+1,i] = alpha[i+1]*u[j+1,i+1] + beta[i+1]

    alpha = np.zeros((N+1))
    beta = np.zeros((N+1))
    alpha[1] = 0
    beta[1] = 0
    for i in range(1,N):
        e2 = -E2((u[j,i] + u[j,i+1]) / 2, (v[j,i] + v[j,i+1]) / 2)
        e2_ = -E2((u[j,i] + u[j,i-1]) / 2, (v[j,i] + v[j,i-1]) / 2)

        A = -1/h * (e2_ / h - alpha2/2 * (p[i]-p[i-1])/h - beta2/2 * (u[j][i]-u[j][i-1])/h )
        B = -1/h * (-e2 / h + alpha2/2 * (p[i+1]-p[i])/h + beta2/2 * (u[j][i+1]-u[j][i])/h - e2_ / h - alpha2/2 * (p[i]-p[i-1])/h + beta2/2 * (u[j][i]-u[j][i-1])/h) - 1/tau
        C = -1/h * (e2 / h + alpha2 /2 * (p[i+1]-p[i])/h + beta2/2 * (u[j][i+1]-u[j][i]/h))
        alpha[i+1] = -C / (A * alpha[i] + B)
        beta[i+1] = (-v[j][i]/tau - eta1*v[j][i]*(1 - (u[j][i] + v[j][i])/p[i]) - A * beta[i]) / (A * alpha[i] + B)

    v[j+1,N] = 0
    for i in range(N-1,0,-1):
        v[j+1,i] = alpha[i+1]*v[j+1,i+1] + beta[i+1]
    if M % 50 == 0:
        pbar.set_description(f"Iteration {j+1}/{M} U.mean: {u[j+1].mean():.5f} V.mean: {v[j+1].mean():.5f}")



Calculating:   0%|          | 0/2000 [00:00<?, ?it/s]

In [34]:
import matplotlib.animation as animation
x = np.linspace(0,2, N+1)
fig, ax = plt.subplots(figsize=(10, 6))
line1, = ax.plot(x, u[0], label='1 популяция')
line2, = ax.plot(x, v[0], label='2 популяция')
ax.set_ylim(0, max(u.max(), v.max()) * 1.2)
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True)

# Animation function
def update(frame):
    line1.set_ydata(u[frame])
    line2.set_ydata(v[frame])
    ax.set_title(f'Time = {frame * tau:.2f}')
    return line1, line2

# Create animation (save every 10th frame to reduce file size)
ani = animation.FuncAnimation(fig, update, frames=range(0, M+1, 10), interval=20)
ani.save(f'population2.gif', writer='pillow', fps=10)
plt.close()